In [6]:
import model as m
import analysis
from cases import generate_test_cases
from tools import get_tool_declarations

Init

In [7]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"

move_values = [0.5, 1, 2, -0.5, -1, -2]
rotate_values = [45, 90, -45, -90]

tool_declarations = get_tool_declarations()
test_cases = generate_test_cases(move_values=move_values, rotate_values=rotate_values)

tokenizer, model = m.initialize(MODEL)
raw_results = m.run_single_call_experiments(model, tokenizer, test_cases)

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 412.92it/s]


TypeError: can only concatenate str (not "dict") to str

Run Experiments

In [ ]:
evaluated_results = []

for result in raw_results:
    evaluation = analysis.evaluate_response(
        output_text=result["raw_output"],
        expected_call=result["expected_call"],
        tool_declarations=tool_declarations,
    )

    evaluated_results.append(
        {
            "prompt": result["prompt"],
            "category": result["category"],
            "expected_call": result["expected_call"],
            "raw_output": result["raw_output"],
            "parsed_call": evaluation["parsed_call"],
            "adherence_pass": evaluation["adherence_pass"],
            "accuracy_pass": evaluation["accuracy_pass"],
            "adherence_reason": evaluation["adherence_reason"],
            "accuracy_reason": evaluation["accuracy_reason"],
        }
    )

Analysis

In [ ]:
adherence_rate = 100 * sum(item["adherence_pass"] for item in evaluated_results) / len(evaluated_results)
accuracy_rate = 100 * sum(item["accuracy_pass"] for item in evaluated_results) / len(evaluated_results)

print(f"Adherence: {adherence_rate:.2f}%")
print(f"Accuracy: {accuracy_rate:.2f}%")

In [ ]:
for item in evaluated_results:
    if not item["adherence_pass"] or not item["accuracy_pass"]:
        print("-" * 80)
        print("Prompt:", item["prompt"])
        print("Expected:", item["expected_call"])
        print("Parsed:", item["parsed_call"])
        print("Adherence:", item["adherence_pass"], "|", item["adherence_reason"])
        print("Accuracy:", item["accuracy_pass"], "|", item["accuracy_reason"])
        print("Raw output:", item["raw_output"])